# LLMOps Assignment 2 — API Cost, Latency and Prompt Version Management
**Course:** MLOps · **Student:** ANDRA RAJESH · **BITS ID:** 2025AE05909
**Scenario:** EduAssist AI - Academic question answering assistant that must return strict JSON.


# 1. Difference Between MLOps and LLMOps

| Aspect | MLOps | LLMOps |
|---|---|---|
| Main focus | Managing trained models and the data pipelines used to train them | Managing prompts, model versions, retrieved information and safety rules |
| What changes frequently | Data and models need to be updated or retrained when there are changes | Prompts and model providers can be changed frequently without retraining the model |
| Cost | Mainly depends on training resources, computing power and model deployment | Mainly depends on API usage, where input and output tokens are charged separately |
| Latency | Depends on model size, hardware and infrastructure | Depends on prompt length, context size, output length, model selection and API response time |
| Evaluation | Usually measured using metrics like accuracy and ROC-AUC on test data | Evaluated based on response quality, correctness, whether the answer is based on the given context, safety and human feedback |
| Versioning | Maintains versions of datasets and trained models | Maintains versions of prompts and models, including prompt changes, history and production versions |
| Common failures | Data changes or drift can reduce model accuracy over time | Changes in prompts or provider models can affect responses, and the model may also generate incorrect formats like invalid JSON |


In [1]:
import json
import urllib.request

import pandas as pd
import mlflow

from eduassist import config
from eduassist.config import get_settings, load_questions
from eduassist.llm_client import LLMClient
from eduassist.workflow import setup_mlflow, task1_task2, task3, write_metadata

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

settings = get_settings()
print("Settings (safe to show - no secrets):")
for k, v in settings.public_dict().items():
    print(f"  {k:18s} {v}")
_ = settings.api_key()
print("api key loaded from environment (hidden)")

if settings.tracking_uri.startswith("http"):
    try:
        urllib.request.urlopen(settings.tracking_uri + "/health", timeout=5)
    except Exception as exc:
        raise RuntimeError("MLflow server not reachable. In a terminal run: "
                           "bash scripts/start_mlflow.sh") from exc
setup_mlflow(settings)
client = LLMClient(settings)
questions = load_questions()
config.OUTPUTS_DIR.mkdir(exist_ok=True)
print("\nMLflow", mlflow.__version__, "| tracking:", settings.tracking_uri,
      "| experiment:", config.EXPERIMENT_NAME)
print("API:", client.api_version)
pd.DataFrame(questions)[["id", "question"]]

2026/11/05 23:05:08 INFO mlflow.tracking.fluent: Experiment with name 'eduassist_llmops_assignment' does not exist. Creating a new experiment.


Settings (safe to show - no secrets):
  provider           gemini
  model              gemini-3.5-flash-lite
  base_url           https://generativelanguage.googleapis.com/v1beta/openai/
  temperature        0.0
  max_tokens         400
  price_input_per_m  0.3
  price_output_per_m 2.5
  currency           USD
  price_date         2026-11-05
  price_source       https://ai.google.dev/gemini-api/docs/pricing
api key loaded from environment (hidden)



MLflow 3.18.0 | tracking: http://127.0.0.1:5000 | experiment: eduassist_llmops_assignment
API: gemini chat.completions (openai-python 3.45.0)


,id,question
0,Q1,What is the difference between supervised and unsupervised learning?
1,Q2,Explain overfitting in machine learning and give two ways to prevent it.
2,Q3,What is the time complexity of binary search and why?
3,Q4,State Newton's second law of motion and give a real-life example.
4,Q5,What is the role of mitochondria in a eukaryotic cell?


## 1 - Calling an LLM and measuring latency and cost
Each of the five questions is sent once with prompt v1. For every request the client records question ID, prompt version, model name (as returned by the API), start/end time, latency and status. Failed calls are retried at most twice on rate limit/server errors, the number of attempts is recorded.

In [2]:
t12 = task1_task2(client, questions, label="v1")
t12["latency_table"]

  Q1 v1: success  5.616s in=21 out=396


  Q2 v1: success  1.802s in=24 out=245


  Q3 v1: success  5.307s in=22 out=485


  Q4 v1: success  8.299s in=27 out=0


  Q5 v1: success  1.571s in=22 out=234


🏃 View run task1_task2_latency_cost at: http://127.0.0.1:5000/#/experiments/1/runs/330ec76ac2ad442085926e2b402f5e4d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


,question_id,prompt_version,model_name,start_time,end_time,latency_seconds,status
0,Q1,v1,gemini-3.5-flash-lite,2026-11-05T17:35:10.525+00:00,2026-11-05T17:35:16.141+00:00,5.6159,success
1,Q2,v1,gemini-3.5-flash-lite,2026-11-05T17:35:43.849+00:00,2026-11-05T17:35:45.651+00:00,1.8016,success
2,Q3,v1,gemini-3.5-flash-lite,2026-11-05T17:35:50.651+00:00,2026-11-05T17:35:55.958+00:00,5.3070,success
3,Q4,v1,gemini-3.5-flash-lite,2026-11-05T17:36:36.592+00:00,2026-11-05T17:36:44.891+00:00,8.2988,success
4,Q5,v1,gemini-3.5-flash-lite,2026-11-05T17:37:45.954+00:00,2026-11-05T17:37:47.525+00:00,1.5715,success


In [3]:
stats = pd.DataFrame([t12["latency_stats"]]).rename(
    columns=lambda c: f"{c}_latency_s").round(4)
stats

,mean_latency_s,median_latency_s,min_latency_s,max_latency_s,std_latency_s
0,4.519,5.307,1.5715,8.2988,2.8365


In [4]:
slow = t12["slowest"]
req = t12["requests"]
print(f"Slowest request: {slow['question_id']} - {slow['latency_seconds']:.3f} s "
      f"({slow['latency_seconds'] / t12['latency_stats']['mean']:.2f} x the mean)")
print(f"Output tokens {slow['output_tokens']} vs mean {req['output_tokens'].mean():.1f}; "
f"Input tokens {slow['input_tokens']} vs mean {req['input_tokens'].mean():.1f}; "
f"Attempts {slow['attempts']}")
print("First request of the session:", slow['question_id'] == req.iloc[0]['question_id'])
print("\nCorrelation latency ~ output tokens:",round(req['latency_seconds'].corr(req['output_tokens']), 3))

Slowest request: Q4 - 8.299 s (1.84 x the mean)
Output tokens 0 vs mean 272.0; Input tokens 27 vs mean 23.2; Attempts 3
First request of the session: False

Correlation latency ~ output tokens: -0.257


**Interpretation.** Mean latency was 4.52 s (median 5.31 s, min 1.57 s, max 8.30 s, std 2.84 s). The slowest request was **Q4 (8.30 s, 1.8× the mean)**. It needed three attempts and returned an empty answer (0 output tokens), so its delay came from the provider side (free-tier throttling / retries, and probably a content filter), not from generating a long answer. For the other questions latency followed output length: Q3 with 485 output tokens took 5.31 s, while Q5 with 234 tokens took 1.57 s. The recorded latency covers only the successful attempt; the 27–61 s gaps between start times show the extra waiting users would see during retries on the free tier.

## 2 - Token counting and cost
Token counts come from the `usage` field of every API response (`prompt_tokens`, `completion_tokens`).
Input cost = input tokens / 1,000,000 × input price per million  
Output cost = output tokens / 1,000,000 × output price per million  
Total cost per request = input cost + output cost  
Projected monthly cost = average cost per request × 10,000

In [5]:
print(f"Model: {settings.model} | API: {client.api_version}")
print(f"Pricing: input {settings.price_input_per_m} / output {settings.price_output_per_m} "
      f"{settings.currency} per 1M tokens | pricing date {settings.price_date} "
      f"| source {settings.price_source}")
ct = t12["cost_table"].copy()
for c in ["input_cost", "output_cost", "total_cost"]:
    ct[c] = ct[c].map(lambda v: f"{v:.8f}")
ct

Model: gemini-3.5-flash-lite | API: gemini chat.completions (openai-python 3.45.0)
Pricing: input 0.3 / output 2.5 USD per 1M tokens | pricing date 2026-11-05 | source https://ai.google.dev/gemini-api/docs/pricing


,question_id,input_tokens,output_tokens,total_tokens,input_cost,output_cost,total_cost
0,Q1,21,396,417,0.00000630,0.00099000,0.00099630
1,Q2,24,245,269,0.00000720,0.00061250,0.00061970
2,Q3,22,485,507,0.00000660,0.00121250,0.00121910
3,Q4,27,0,27,0.00000810,0.00000000,0.00000810
4,Q5,22,234,256,0.00000660,0.00058500,0.00059160


In [6]:
avg = t12["avg_cost"]
monthly = t12["monthly_cost"]
print(f"Average cost per request : {avg:.8f} {settings.currency}")
print(f"Projected monthly cost   : {monthly:.4f} {settings.currency} for 10,000 requests")
r = t12["requests"]
in_share = r["input_cost"].sum() / max(r["total_cost"].sum(), 1e-12)
print(f"Share of cost from input tokens: {in_share:.0%}, output tokens: {1 - in_share:.0%}")

# Effect of shorter answers on the monthly bill
half_out = (r["input_cost"] + r["output_cost"] / 2).mean() * 10_000
print(f"What-if responses were 50% shorter      : {half_out:.4f} {settings.currency}/month")

Average cost per request : 0.00068696 USD
Projected monthly cost   : 6.8696 USD for 10,000 requests
Share of cost from input tokens: 1%, output tokens: 99%
What-if responses were 50% shorter      : 3.4696 USD/month


**Cost-optimisation recommendation.** With gemini-3.5-flash-lite (USD 0.30 per 1M input, USD 2.50 per 1M output tokens, published paid-tier price read on 2026-11-05) the average cost was **USD 0.00068696 per request → USD 6.87 per month for 10,000 requests**. About **99 % of the cost comes from output tokens** (1,360 output vs 116 input tokens, and output is 8.3× more expensive). The most effective saving is therefore to **shorten the responses** – an explicit length limit in the prompt (e.g. "answer in at most 100 words") plus a lower `max_tokens`. Halving the output length would reduce the projection to about USD 3.47 per month. Task 3 confirms this: the concise JSON prompt v3 averaged only 94 output tokens and projects to USD 2.61 per month, 64 % below v1. Prompt caching would help little here because input tokens are a very small share of the cost.

## 3 - Prompt versioning and MLflow evaluation

1. registers v1, v2, v3 in the **MLflow Prompt Registry** under one prompt name with commit messages (re-running does not create duplicates);
2. runs each version on the same five questions in its own MLflow run (prompt loaded from the registry inside the run, which links prompt and run);
3. logs `avg_latency_seconds`, `avg_input_tokens`, `avg_output_tokens`, `avg_cost_per_request`, `valid_json` (+ schema validity, keyword-coverage quality score, projected monthly cost);
4. runs `mlflow.genai.evaluate` with boolean scorers `valid_json` and `schema_valid` — MLflow aggregates the per-response feedback into a mean = pass rate;
5. promotes the best version to the alias **`production`**.

`valid_json = valid JSON responses / total responses` (strict: the whole response must parse as a JSON object — code fences or extra text count as invalid).

In [7]:
t3 = task3(client, questions)
for label, p in t3["registered"].items():
    print(f"{label}: registry version {p.version} | commit: {p.commit_message} | "
          f"tags: {p.tags}")

2026/11/05 23:07:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist-academic-qa, version 1


2026/11/05 23:07:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist-academic-qa, version 2


2026/11/05 23:07:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist-academic-qa, version 3


Evaluating prompt v1 ...


  Q1 v1: error_503 3.313s in=0 out=0


  Q2 v1: success  2.546s in=24 out=273


... (13 outputs left out)

  Q3 v3: success  6.155s in=84 out=103


  Q4 v3: success  5.346s in=89 out=0


  Q5 v3: success  2.825s in=84 out=130


v1: registry version 1 | commit: v1 basic: minimal instruction, free-text answer (baseline) | tags: {'style': 'basic', 'output_format': 'free_text', 'label': 'v1', 'project': 'EduAssist AI'}
v2: registry version 2 | commit: v2 structured: academic-assistant role, concise, undergraduate-level language | tags: {'style': 'structured', 'output_format': 'free_text', 'label': 'v2', 'project': 'EduAssist AI'}
v3: registry version 3 | commit: v3 json: strict JSON output with answer, confidence, key_points schema for downstream parsing | tags: {'style': 'json', 'output_format': 'json', 'label': 'v3', 'project': 'EduAssist AI'}


In [8]:
for name, diff in t3["diffs"].items():
    print(f"===== diff {name} =====")
    print(diff, "\n")

===== diff v1->v2 =====
--- v1 (registry v1)
+++ v2 (registry v2)
@@ -1,2 +1,4 @@
-Answer the following student question clearly.
+You are an academic assistant.
+Answer the following student question in a clear, concise, and accurate manner.
+Use simple language suitable for undergraduate students.
 Question: {{question}} 

===== diff v2->v3 =====
--- v2 (registry v2)
+++ v3 (registry v3)
@@ -1,4 +1,10 @@
 You are an academic assistant.
-Answer the following student question in a clear, concise, and accurate manner.
-Use simple language suitable for undergraduate students.
+Answer the student question and return ONLY valid JSON.
+Do not include markdown, explanations outside JSON, or code fences.
+Use this exact JSON schema:
+{
+"answer": "string",
+"confidence": "high | medium | low",
+"key_points": ["string"]
+}
 Question: {{question}} 



In [9]:
t = t3["table"]
comparison = pd.DataFrame({
    "Prompt Version": t["prompt_version"],
    "Avg Latency (s)": t["avg_latency_seconds"].round(3),
    "Avg Input Tokens": t["avg_input_tokens"].round(1),
    "Avg Output Tokens": t["avg_output_tokens"].round(1),
    f"Avg Cost ({settings.currency})": t["avg_cost_per_request"].map(lambda v: f"{v:.8f}"),
    "Valid JSON Rate": t["valid_json"].round(2),
    "Schema Valid Rate": t["schema_valid_rate"].round(2),
    "Quality (keyword coverage)": t["quality_keyword_coverage"].round(2),
    f"Monthly cost 10k ({settings.currency})": t["projected_monthly_cost_10k"].round(4),
    "Recommended?": t["recommended"],
})
comparison

,Prompt Version,Avg Latency (s),Avg Input Tokens,Avg Output Tokens,Avg Cost (USD),Valid JSON Rate,Schema Valid Rate,Quality (keyword coverage),Monthly cost 10k (USD),Recommended?
0,v1,2.400,23.8,290.0,0.00073213,0.0,0.0,0.80,7.3212,no
1,v2,6.788,47.2,213.2,0.00054716,0.0,0.0,0.85,5.4716,no
2,v3,8.155,85.2,94.0,0.00026056,0.8,0.8,0.80,2.6056,yes


In [10]:
prod = mlflow.genai.load_prompt(f"prompts:/{config.PROMPT_NAME}@{config.PRODUCTION_ALIAS}")
print(f"Alias '{config.PRODUCTION_ALIAS}' -> {config.PROMPT_NAME} version {prod.version}")
print(prod.template)

Alias 'production' -> eduassist-academic-qa version 3
You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
"answer": "string",
"confidence": "high | medium | low",
"key_points": ["string"]
}
Question: {{question}}


In [11]:
d = t3["detail"]
for qid in ["Q1"]:
    for _, row in d[d["question_id"] == qid].iterrows():
        print(f"----- {qid} {row['prompt_version']} | valid_json={row['valid_json']} "
              f"| tokens out={row['output_tokens']} -----")
        print(row["response_text"][:700], "\n")

----- Q1 v1 | valid_json=False | tokens out=0 -----
 

----- Q1 v2 | valid_json=False | tokens out=244 -----
Here is the difference between supervised and unsupervised learning, broken down simply:

*   **Supervised Learning** is like learning with a teacher. You provide the computer with **labeled data**—meaning the data comes with the correct answers. The goal is for the computer to learn the relationship between the input data and the correct output so it can predict answers for new, unseen data. 
    *   *Real-world example:* Teaching an algorithm to recognize emails as "Spam" or "Not Spam" by showing it thousands of emails that have already been labeled by humans.

*   **Unsupervised Learning** is like learning on your own. You provide the computer with **unlabeled data**—data without any pred 

----- Q1 v3 | valid_json=True | tokens out=107 -----
{
"answer": "Supervised learning uses labeled data to train models to predict outcomes or classify information, whereas unsupervised le

In [12]:
b = t3["best"]
others = t[t["prompt_version"] != b["prompt_version"]]
print("DRAFT facts for the promotion justification (rewrite in your own words, >= 150 words):")
print(f"- Promoted: {b['prompt_version']} (registry version {b['registry_version']}).")
print(f"- valid_json {b['valid_json']:.2f}, schema valid {b['schema_valid_rate']:.2f}; "
      f"others: " + ", ".join(f"{r.prompt_version}={r.valid_json:.2f}" for r in others.itertuples()))
print(f"- quality (keyword coverage) {b['quality_keyword_coverage']:.2f}; others: "
      + ", ".join(f"{r.prompt_version}={r.quality_keyword_coverage:.2f}" for r in others.itertuples()))
print(f"- avg latency {b['avg_latency_seconds']:.3f}s; avg tokens in/out "
      f"{b['avg_input_tokens']:.0f}/{b['avg_output_tokens']:.0f}")
print(f"- avg cost {b['avg_cost_per_request']:.8f} {settings.currency}; monthly (10k) "
      f"{b['projected_monthly_cost_10k']:.4f} {settings.currency}")

DRAFT facts for the promotion justification (rewrite in your own words, >= 150 words):
- Promoted: v3 (registry version 3).
- valid_json 0.80, schema valid 0.80; others: v1=0.00, v2=0.00
- quality (keyword coverage) 0.80; others: v1=0.80, v2=0.85
- avg latency 8.155s; avg tokens in/out 85/94
- avg cost 0.00026056 USD; monthly (10k) 2.6056 USD


### Promotion justification
EduAssist needs answers that the application can parse, so JSON validity is the deciding criterion. Prompts v1 and v2 never produced JSON (valid_json = 0.0), while **v3 returned valid, schema-conformant JSON for 4 of 5 questions (valid_json = 0.8)**. Its single failure was an empty response for Q4 – the same question that also came back empty in Task 1 – not malformed JSON. Answer quality was comparable across versions: keyword coverage 0.80 for v1 and v3 and 0.85 for v2, and v3 additionally provides a confidence level and key points. v3 uses the most input tokens (85 vs 24–47) because of the schema instructions, but it is the **cheapest** version: only 94 output tokens on average instead of 213–290, i.e. USD 0.00026 per request and USD 2.61 per month versus USD 5.47–7.32. Its weakness is latency (median 6.2 s vs about 2.3 s), partly caused by free-tier retries – three of five v3 requests needed several attempts. Because EduAssist is not a real-time chat, correct format and lower cost outweigh a few seconds. **v3 is therefore promoted to `production`**, with a guardrail before go-live (JSON response mode, retry on empty or invalid output, monitoring of valid_json), because 0.8 is not yet 1.0.

In [13]:
meta = write_metadata(client)
print(json.dumps(meta, indent=2))
print("\nFiles in outputs/:")
for p in sorted(config.OUTPUTS_DIR.iterdir()):
    print("  ", p.name)

{
  "provider": "gemini",
  "model": "gemini-3.5-flash-lite",
  "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
  "temperature": 0.0,
  "max_tokens": 400,
  "price_input_per_m": 0.3,
  "price_output_per_m": 2.5,
  "currency": "USD",
  "price_date": "2026-11-05",
  "price_source": "https://ai.google.dev/gemini-api/docs/pricing",
  "api_version": "gemini chat.completions (openai-python 3.45.0)",
  "mlflow_version": "3.18.0",
  "experiment": "eduassist_llmops_assignment",
  "prompt_name": "eduassist-academic-qa",
  "run_utc": "2026-11-05T17:42:32+00:00"
}

Files in outputs/:
   .gitkeep
   latency_summary.csv
   latency_tokens_cost.csv
   prompt_diff_v1_v2.txt
   prompt_diff_v2_v3.txt
   prompt_evaluation_requests.csv
   prompt_evaluation_results.csv
   run_metadata.json


## Conclusion
Using gemini-3.5-flash-lite through the Gemini OpenAI-compatible API, I measured latency, token usage and cost for every request and projected USD 6.87 per month for 10,000 requests. Three prompt versions were registered in the MLflow Prompt Registry, evaluated on the same five questions and compared; **v3 (JSON output) was promoted to the alias `production`** because it is the only version that produces machine-readable output and also the cheapest. Limitations: only five questions and one run per version (sensitive to outliers such as the 25 s v2-Q4 request and to free-tier throttling), two empty Q4 responses whose finish reason was not logged, and keyword coverage is only a rough quality measure.